In [ ]:
import pandas as pd
import numpy as np

print("Libraries imported successfully.")

Libraries imported successfully.


In [2]:
DATA_PATH = "../data/processed/ml_dataset.csv"

df = pd.read_csv(DATA_PATH)

print("Dataset loaded successfully.")
print("Shape:", df.shape)

Dataset loaded successfully.
Shape: (65499, 21)


In [3]:
import joblib

MODEL_PATH = "../models/improved_random_forest_glucose_model.pkl"
FEATURE_PATH = "../models/feature_columns.pkl"

model = joblib.load(MODEL_PATH)
feature_columns = joblib.load(FEATURE_PATH)

print("Model loaded successfully.")
print("Number of features:", len(feature_columns))

Model loaded successfully.
Number of features: 18


In [4]:
candidates = df[
    (
        (df["meal_carbs"] >= 30) |
        (df["exercise_duration"] > 0) |
        (df["sleep_duration_hours"] > 0)
    )
].copy()

sample = candidates.sort_values(
    by=[
        "meal_carbs",
        "exercise_duration",
        "sleep_duration_hours"
    ],
    ascending=False
).iloc[0].copy()

print("Patient:", sample["Patient"])
print("Timestamp:", sample["timestamp"])
print()
print("Current glucose:", sample["glucose_value"])
print("Meal carbs:", sample["meal_carbs"])
print("Exercise:", sample["exercise_duration"], "minutes")
print("Sleep:", round(sample["sleep_duration_hours"], 2), "hours")
print("Sleep quality:", sample["sleep_quality"])

Patient: 544-ws-training
Timestamp: 2027-05-28 17:55:00

Current glucose: 91
Meal carbs: 162.0
Exercise: 0.0 minutes
Sleep: 8.72 hours
Sleep quality: 2.0


In [5]:
def predict_glucose(state):
    X = pd.DataFrame([state])[feature_columns]
    return model.predict(X)[0]

baseline_prediction = predict_glucose(sample)

print(
    "Predicted glucose after 30 minutes:",
    round(baseline_prediction, 2),
    "mg/dL"
)

Predicted glucose after 30 minutes: 94.7 mg/dL


In [6]:
def glucose_status(glucose):
    if glucose < 70:
        return "LOW"
    elif glucose < 140:
        return "NORMAL"
    elif glucose < 180:
        return "ELEVATED"
    else:
        return "HIGH"

status = glucose_status(baseline_prediction)

print("Predicted glucose status:", status)

Predicted glucose status: NORMAL


In [7]:
observations = []

if sample["meal_carbs"] >= 50:
    observations.append(
        "High carbohydrate intake detected."
    )

if sample["exercise_duration"] == 0:
    observations.append(
        "No exercise recorded in the current state."
    )

if sample["sleep_quality"] <= 2:
    observations.append(
        "Sleep quality is relatively low."
    )

if sample["sleep_duration_hours"] < 6:
    observations.append(
        "Sleep duration is relatively short."
    )

if sample["glucose_change_15min"] >= 15:
    observations.append(
        "Glucose is rising rapidly."
    )

if sample["glucose_change_15min"] <= -15:
    observations.append(
        "Glucose is falling rapidly."
    )

if len(observations) == 0:
    observations.append(
        "No major lifestyle concerns detected."
    )

print("Lifestyle observations:")
for observation in observations:
    print("-", observation)

Lifestyle observations:
- High carbohydrate intake detected.
- No exercise recorded in the current state.
- Sleep quality is relatively low.


In [8]:
recommendations = []

if sample["meal_carbs"] >= 50:
    recommendations.append(
        "Consider a lower-carbohydrate meal option."
    )

if sample["exercise_duration"] == 0:
    recommendations.append(
        "Consider maintaining appropriate physical activity."
    )

if sample["sleep_quality"] <= 2:
    recommendations.append(
        "Consider improving sleep quality and maintaining a consistent sleep routine."
    )

if sample["sleep_duration_hours"] < 6:
    recommendations.append(
        "Consider getting adequate sleep."
    )

if not recommendations:
    recommendations.append(
        "Maintain the current lifestyle pattern."
    )

print("Personalized recommendations:")
for recommendation in recommendations:
    print("-", recommendation)

Personalized recommendations:
- Consider a lower-carbohydrate meal option.
- Consider maintaining appropriate physical activity.
- Consider improving sleep quality and maintaining a consistent sleep routine.


In [9]:
print("=" * 55)
print("       DIA DECISION AI — PERSONALIZED ANALYSIS")
print("=" * 55)

print()
print("Patient:", sample["Patient"])
print("Current glucose:", sample["glucose_value"], "mg/dL")
print(
    "Predicted glucose (30 min):",
    round(baseline_prediction, 2),
    "mg/dL"
)
print("Predicted status:", status)

print()
print("Lifestyle observations:")

for observation in observations:
    print("•", observation)

print()
print("Personalized recommendations:")

for recommendation in recommendations:
    print("•", recommendation)

print()
print("=" * 55)

       DIA DECISION AI — PERSONALIZED ANALYSIS

Patient: 544-ws-training
Current glucose: 91 mg/dL
Predicted glucose (30 min): 94.7 mg/dL
Predicted status: NORMAL

Lifestyle observations:
• High carbohydrate intake detected.
• No exercise recorded in the current state.
• Sleep quality is relatively low.

Personalized recommendations:
• Consider a lower-carbohydrate meal option.
• Consider maintaining appropriate physical activity.
• Consider improving sleep quality and maintaining a consistent sleep routine.

